# 03 — Load a 3D Revision

Companion to [Chapter 09](../09-3d.md). This project is DMS-only, so the 3D
model/revision/node APIs are called as **raw HTTP** through `client.get` /
`client.post` rather than typed SDK methods -- good practice for the Document Parser
API in the next chapter, which is raw HTTP for a different reason (it's internal).

Cell order: auth -> find/create model shell -> find/create revision -> poll with a
timeout -> inspect nodes -> map a couple of names to assets -> bridge to the Function.

In [ ]:
YOURNAME = "YOURNAME"  # [CHANGE]

import time
from cognite.client import CogniteClient

client = CogniteClient()
space = f"isp_{YOURNAME}_TRN"
model_name = f"trd_{YOURNAME}_TRN_CAD"
file_xid = f"file_{YOURNAME}_TRN_3D_21_SEP"
project = client.config.project


## Step 1 -- find or create the 3D model shell

On a DMS-only project the model shell needs `space` + `type: CAD` -- fields the
classic 3D loaders don't expect, which is why this isn't a typed SDK call.

In [ ]:
base_models = f"/api/v1/projects/{project}/3d/models"
payload = client.get(base_models, params={"limit": 1000}).json()
model = next((m for m in payload.get("items", []) if m.get("name") == model_name), None)

if model is None:
    resp = client.post(base_models, json={"items": [{"name": model_name, "space": space, "type": "CAD"}]})
    model = resp.json()["items"][0]

model_id = model["id"]
print("model id:", model_id, "name:", model["name"])

## Step 2 -- find or create a revision from the classic OBJ file

In [ ]:
base_revisions = f"{base_models}/{model_id}/revisions"
revisions = client.get(base_revisions, params={"limit": 100}).json().get("items", [])

if revisions:
    revision = revisions[0]
    print("found existing revision:", revision["id"], "status:", revision.get("status"))
else:
    src = client.files.retrieve(external_id=file_xid)
    assert src is not None and src.uploaded, f"OBJ classic file {file_xid} missing or not uploaded"
    session = client.iam.sessions.create()
    resp = client.post(base_revisions, json={"items": [{"fileId": src.id, "published": True, "nonce": session.nonce}]})
    revision = resp.json()["items"][0]
    print("created revision:", revision["id"])

revision_id = revision["id"]

## Step 3 -- poll with a timeout (`Queued` / `Processing` -> `Done` | `Failed`)

Conversion can take minutes on a real model. This cell has a bounded budget and will
hand control back rather than block forever -- rerun the cell later if it times out.

In [ ]:
deadline = time.time() + 20 * 60
status = revision.get("status")
while status in ("Queued", "Processing") and time.time() < deadline:
    time.sleep(15)
    revision = client.get(f"{base_revisions}/{revision_id}").json()
    status = revision.get("status")
    print("status:", status)

print("final status:", status)
if status not in ("Done",):
    print("Not done yet (or Failed) -- rerun this cell later rather than creating a second revision.")

## Step 4 -- publish (if `Done` and not already published), then list nodes

In [ ]:
if status == "Done" and not revision.get("published"):
    body = {"items": [{
        "id": revision_id,
        "instanceId": {"space": space, "externalId": f"cog_3d_revision_{revision_id}"},
        "update": {"published": {"set": True}},
    }]}
    resp = client.post(f"{base_revisions}/update", json=body)
    revision = resp.json()["items"][0]
    print("published:", revision.get("published"))

nodes = []
if status == "Done":
    nodes = client.get(f"{base_revisions}/{revision_id}/nodes", params={"limit": 1000}).json().get("items", [])
    print(f"{len(nodes)} nodes; example names: {sorted({n.get('name') for n in nodes if n.get('name')})[:10]}")

## Step 5 -- inspect one mapping by hand (pump A)

The full Function maps five tags plus the deck; here, just look at one node to see
the shape of what you're working with before the Function does all of them.

In [ ]:
pump_node = next((n for n in nodes if n.get("name") == "21-PA-2001A"), None)
print("pump node:", pump_node)

## Bridge to the Function

Package this into `Load3DRevision`: the full `TAG_MAP` for all five tags plus the
deck, the `Cognite3DObject`/`CogniteCADNode`/`Cognite3DModel`/`Cognite3DRevision`/
`CogniteCADRevision` node writes that make the asset's 3D tab actually render, and
the same resume-not-restart discipline (checking for an existing revision before
creating a new one) -- but as something you can safely call repeatedly from a
workflow, not something you babysit interactively. See
[Chapter 09, section 9.4](../09-3d.md#94-write-the-function-load3drevision).